# Grounded answers with the Citations API

The Claude **Citations** feature grounds an answer in documents you supply. Enable citations on
`document` content blocks, ask a question, and Claude returns its answer as a sequence of text
blocks — each block is either plain prose or a **claim carrying a `citations` list** that points
at the exact source text (`cited_text`) and its location.

This notebook builds a small knowledge base, asks a question, inspects the citation objects, and
renders the cited answer as a footnoted page.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root).

In [ ]:
# Setup
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

# Shared helpers live in _citations.py. Make it importable whether the working directory is this
# folder or the repo root.
for _p in (".", "citations"):
    if os.path.isfile(os.path.join(_p, "_citations.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _citations import (
    DEFAULT_QUESTION,
    WILDLIFE_DOCS,
    ask,
    build_content,
    location_label,
    render_html,
    text_blocks,
)

load_dotenv()
client = Anthropic()

## 1. Documents + the request

Each document becomes a `document` content block with `citations: {"enabled": True}`. Plain-text
documents are auto-chunked into sentences, so Claude can cite one sentence or a span of them. We
also put `cache_control` on each document — citations and prompt caching compose (short demo docs
won't actually cache, but the pattern is correct). `build_content` assembles the message; here is
what one document block looks like:

In [ ]:
content_blocks = build_content(WILDLIFE_DOCS, DEFAULT_QUESTION)
print("documents:", [d["title"] for d in WILDLIFE_DOCS])
print("question:", DEFAULT_QUESTION)
content_blocks[0]  # the first document block

## 2. Ask, and inspect the citations

`ask` sends the documents + question and returns the response content blocks. Every text block
that carries a `citations` list is a claim Claude grounded in a source. Notice that `cited_text`
is the *exact* quote from the document, and the character indices point back into it.

In [ ]:
content = ask(client, WILDLIFE_DOCS, DEFAULT_QUESTION)

for block in text_blocks(content):
    for cit in block["citations"]:
        print(f"CLAIM:  {block['text']!r}")
        print(f"  source: [{cit['document_title']}] {location_label(cit)}")
        print(f"  quote : {cit['cited_text']!r}\n")

## 3. Render it as a footnoted page

`render_html` turns the response into an editorial page: each cited claim gets a red footnote
marker (hover to preview the source, click to jump to the reference). This is the same markup a
hand-built footnoted article uses — but every citation here came from the API.

In [ ]:
from IPython.display import HTML

HTML(render_html(DEFAULT_QUESTION, content))

## Notes

- **Document types & citation locations:** plain text → character ranges (`char_location`),
  PDFs → page ranges (`page_location`), custom content blocks → block ranges
  (`content_block_location`). Use custom content when you want to control citation granularity
  (e.g. bullet points or transcript lines) instead of automatic sentence chunking.
- **`cited_text` is free:** it does not count toward output tokens (or input tokens when passed
  back on later turns).
- **Citations must be all-or-none** across the documents in a request.
- **Incompatible with Structured Outputs:** enabling citations together with
  `output_config.format` returns a 400 — citations interleave citation blocks with text, which
  the strict JSON schema can't express.
- **Composes with prompt caching:** cache the source documents with `cache_control` on the
  document blocks; the citation blocks in the response are not themselves cached.